# Deploying NVIDIA Nemotron 3.5 Super VL with TensorRT-LLM

This notebook will walk you through how to run NVIDIA Nemotron 3.5 Super VL with TensorRT-LLM.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs.

NVIDIA Nemotron 3.5 Super VL is available as two checkpoints:

- **BF16**: [`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16)
- **NVFP4**: [`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4)

**This notebook runs the NVFP4 checkpoint on 1x NVIDIA H200 at W4A16.**

**Model size:** 120B total parameters, 12B active. A Mamba-2, MoE, and attention hybrid backbone with a [C-RADIO v4-H](https://huggingface.co/nvidia/C-RADIOv4-H) vision encoder and an MLP projector.

**Context length:** up to 1M tokens

**Inputs:** text, images, and video. Output is text.

Prerequisites for this notebook:
- 1x NVIDIA H200 with recent drivers
- [Docker](https://docs.docker.com/engine/install/) with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html)
- Python 3.10+

## Overview

- **Serve** the Nemotron 3.5 Super VL NVFP4 checkpoint using TensorRT-LLM
- **Query the model** through an OpenAI-compatible API
- **Send images and video** alongside text in the same request
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Decoding options for this model** - Base, MTP, and LiLiCorr
2. **Multimodal inputs** - Image and video formats, limits, and frame sampling
3. **Environment setup** - Container image, client dependencies, and GPU check
   - Launch on NVIDIA Brev
   - Pull the TensorRT-LLM Docker image
   - Install notebook client dependencies
   - Verify GPU
4. **OpenAI-compatible server** - Launch TensorRT-LLM and confirm it is ready
   - Launch the Docker container
   - Create a YAML file
   - Configuration reference
   - Load the model
   - Wait for the server to be ready
5. **Generate responses** - Chat completions, multimodal inference, and tool calling
   - Client setup
   - Single completion
   - Sequential completions
   - Streamed generation
   - Reasoning
   - Document understanding
   - Video understanding
   - Tool calling
   - Controlling reasoning budget
6. **Cleanup and shutdown** - Free the GPU and reset the kernel

## Decoding options for this model

Nemotron 3.5 Super VL can produce tokens three ways. All three serve the same weights and differ only in how many tokens come out of a single forward pass. The base option decodes one token per pass - the other two add speculative decoding, where a cheap draft proposes several tokens ahead, the model verifies them all in a single pass, and every token up to the first mismatch is kept. A rejected token invalidates itself and everything after it, so the payoff depends on how often drafts are right - a bad guess costs compute without producing output.

| Option | Where drafts come from | Extra weights to download |
|---|---|---|
| **Base** (no speculative decoding) | nothing, one token per pass | none |
| **MTP** | a prediction layer inside the checkpoint | a separate MTP checkpoint at W4A16 |
| **LiLiCorr** | a separate draft model | LiLiCorr checkpoint |

TensorRT-LLM supports MTP for this checkpoint. At W4A4 it drafts from the checkpoint itself, while at W4A16 it loads the prediction layer as a separate checkpoint and needs that checkpoint's ID. LiLiCorr, which drafts from a separate model, is covered in the vLLM cookbook's 1x DGX Spark configuration instead.

**Concurrency decides whether speculation pays off.** With few requests in flight the GPU has spare capacity, and speculation spends it to shorten the critical path, which lowers per-request latency. Under heavy load the GPU is already busy producing real tokens, so verifying drafts that end up rejected takes throughput away from queued work. `max_batch_size` caps how many requests can be in flight, so it is the flag that decides which side of that tradeoff you land on.

**Draft length is the main knob.** `max_draft_len` sets how far ahead to guess. Raising it improves the best case per step but lowers the odds that the whole run is accepted, and wastes more compute when it is not - lowering it shrinks the win but makes it more consistent.

**Two caches share the GPU.** Attention layers use a KV cache that grows with sequence length, while the Mamba layers keep a fixed-size recurrent state per sequence. The KV cache is set to FP8 here by `kv_cache_config.dtype`, and the Mamba state is kept in float32 by `kv_cache_config.mamba_ssm_cache_dtype`. `free_gpu_memory_fraction` bounds how much of the GPU the two may claim in total.

This notebook uses **base decoding**, the one path validated on the container above.

## Multimodal inputs

Nemotron 3.5 Super VL accepts text, images, and video in the same conversation. Multimodal content is sent as a list of parts in the `content` field of a message, following the OpenAI chat format - see **Document understanding** and **Video understanding** below for runnable examples.

| Modality | Request part | Format and limits |
|---|---|---|
| Text | `{"type": "text", "text": ...}` | Up to 1M tokens |
| Image | `{"type": "image_url", "image_url": {"url": ...}}` | RGB JPEG or PNG. 512x512 for simple images, 1536x1024 for documents |
| Video | `{"type": "video_url", "video_url": {"url": ...}}` | MP4, 1-2 minutes |

URLs can be public `http(s)` links, `data:` URLs carrying base64-encoded bytes, or `file://` paths for local media.

**Images per request.** One image for simple and document cases. A multi-document request can batch up to 32 page images in a single call.

**PDFs are not accepted.** Render each page to PNG first - for example with `pymupdf` - and send the pages as images.

### Video frame sampling

Frame sampling is a server-launch setting rather than a request parameter. Pick the profile matching the footage you expect:

| Source resolution | fps | num_frames | Flag |
|---|---|---|---|
| **1080p** | 1 | up to 128 | `--media_io_kwargs '{"video": {"fps": 1, "num_frames": 128}}'` |
| **720p and below** | 2 | up to 256 | `--media_io_kwargs '{"video": {"fps": 2, "num_frames": 256}}'` |

The `trtllm-serve` command in this notebook does not set the flag, so video falls back to TensorRT-LLM's default frame count rather than either profile.

## Environment setup

### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this notebook on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3KTfx0rihxrkQWeXE9wALLS3KN0)

### Pull the TensorRT-LLM Docker image

```shell
docker pull nvcr.io/nvidia/tensorrt-llm/release:1.4.0rc0.dev202610040000
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `jinja2` renders the chat template it applies.

In [1]:
# Bootstrap pip only if the kernel environment is missing it
import importlib.util, subprocess, sys

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

%pip install -q openai==2.38.0 transformers==5.9.0 "jinja2>=3.1.0" "ipywidgets>=8.0.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Verify GPU

Confirm your GPUs are visible on the host before starting the Docker container.

> **Expected output:** One row per GPU with its name, total memory, and the host driver version. If `nvidia-smi` is not found, the NVIDIA driver is not installed.

In [2]:
# Confirm the GPUs are visible on the host
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

index, name, memory.total [MiB], driver_version
0, NVIDIA H200, 143771 MiB, 580.173.02


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using TensorRT-LLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the TensorRT-LLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  nvcr.io/nvidia/tensorrt-llm/release:1.4.0rc0.dev202610040000
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

Run the `trtllm-serve` command below from inside this container.

### Create a YAML file

TensorRT-LLM takes most of its configuration from a file rather than flags. Write this inside the container before serving; `trtllm-serve` reads it at startup.

```shell
cat > ./extra-llm-api-config.yml << 'EOF'
tensor_parallel_size: 1
moe_expert_parallel_size: 1
trust_remote_code: true
dtype: bfloat16

enable_chunked_prefill: true
disable_overlap_scheduler: false
max_batch_size: 16
max_num_tokens: 8192
max_seq_len: 131072
max_input_len: 131072

moe_config:
  backend: MARLIN
nvfp4_gemm_config:
  allowed_backends: [marlin]

kv_cache_config:
  dtype: fp8
  mamba_ssm_cache_dtype: float32
  mamba_ssm_stochastic_rounding: false
  free_gpu_memory_fraction: 0.6
  enable_block_reuse: true
  use_kv_cache_manager_v2: true
  mamba_state_config:
    enable_branch_snapshot: true

cuda_graph_config:
  enable_padding: false
  max_batch_size: 16
  batch_sizes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
EOF
```

A few entries worth noting:

- **`dtype: bfloat16`** with the NVFP4 checkpoint is the W4A16 path.
- **`moe_config.backend: MARLIN`** with `nvfp4_gemm_config.allowed_backends: [marlin]` selects the Marlin kernels used on Hopper.
- **`mamba_ssm_cache_dtype: float32` with `mamba_ssm_stochastic_rounding: false`** keeps the Mamba state at full precision, which differs from the vLLM and SGLang settings in this series.
- **`cuda_graph_config.batch_sizes`** enumerates every size from 1 to 16 with padding off, so each batch size gets its own captured graph.

### Configuration reference

Settings for the configuration this notebook runs.

| Setting | Value | Description |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4` | The NVFP4 checkpoint |
| **Served model name** | `nemotron-3.5-super` | What the client cells send |
| **Hardware** | 1x H200 | |
| **Docker image** | `nvcr.io/nvidia/tensorrt-llm/release:1.4.0rc0.dev202610040000` | |
| **Decoding** | Base, one token per forward pass | No `speculative_config` block |
| **Remote code** | Trusted | The checkpoint ships its own modeling code |
| **Tensor parallel size** | 1 | |
| **MoE expert parallel size** | 1 | |
| **Precision** | W4A16 via Marlin | `dtype: bfloat16` is the activation dtype |
| **Max sequence length** | 131072 | Context window, input and output |
| **Max batch size** | 16 | Caps how many requests are in flight |
| **Max num tokens** | 8192 | Caps prefill work per step |
| **KV cache dtype** | FP8 | |
| **GPU memory fraction** | 0.6 | Share of GPU memory for the KV cache |
| **Block reuse** | Enabled, with KV cache manager v2 | Reuses shared conversation prefixes |
| **Mamba state cache** | float32, no stochastic rounding, branch snapshot on | Settings for the recurrent state the Mamba layers keep |
| **Chunked prefill** | Enabled | |
| **Reasoning parser** | `nemotron-v3` | Splits thinking from the final answer |
| **Tool parser** | `qwen3_coder` | Turns the template's tool XML into OpenAI `tool_calls` |
| **Host / port** | `0.0.0.0:8000` | Matches `--network=host` |

### Load the model

Run the following from inside the Docker terminal, in the same directory as the YAML file.

> **Note:** The first launch takes a while. The progress readout can sit at `0%` for several minutes at a time: first while the weights download from Hugging Face and load, then while the server compiles CUDA kernels. That is expected rather than a hang, so give it time instead of restarting.

> **Note:** Parser names are backend-specific: TensorRT-LLM's `nemotron-v3` is `nemotron_v3` in vLLM and `nemotron_3` in SGLang.

```shell
# OpenCV decodes the video inputs used later in this notebook
pip install -q opencv-python-headless

trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4 \
  --config extra-llm-api-config.yml \
  --host 0.0.0.0 \
  --port 8000 \
  --served_model_name nemotron-3.5-super \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder
```

> **Note:** To serve weights already staged on disk, replace the checkpoint with the local directory and set `HF_HUB_OFFLINE=1`.

### Wait for the server to be ready

In a new terminal, poll `/v1/models`:

```shell
TIMEOUT=${TIMEOUT:-3600}   # seconds to wait before giving up
deadline=$(( $(date +%s) + TIMEOUT ))
ready=0

while [ "$(date +%s)" -lt "$deadline" ]; do
  if curl -sf http://localhost:8000/v1/models | grep -q nemotron-3.5-super; then
    ready=1; break
  fi
  echo "Waiting for server..."; sleep 10
done

if [ "$ready" = 1 ]; then
  echo "Server is ready"
else
  echo "Server did not come up within ${TIMEOUT}s. In the terminal running the server, look for:"
  echo "  access    - 401 or 403 from huggingface.co, or a gated-repo message"
  echo "  download  - the weight download still in progress, or a stalled transfer"
  echo "  memory    - an out-of-memory error while allocating the KV cache"
  echo "  startup   - a Python traceback, or CUDA out of memory"
fi
```

> **Note:** The loop gives up after `TIMEOUT` seconds, one hour by default. Set `TIMEOUT` before running it to wait longer or fail faster.

Then check what the server is actually serving, and send one short request to confirm it generates:

```shell
curl -s http://localhost:8000/v1/models | python3 -m json.tool

curl -sf http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "nemotron-3.5-super", "prompt": "Hello", "max_tokens": 16}'
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then `Server is ready` once `nemotron-3.5-super` is listed by `/v1/models`. If it times out instead, it prints the four things to check in the server terminal. The final command returns a short JSON completion, confirming the model is loaded and generating.

## Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, video inference, tool calling, and reasoning budget examples.

Recommended sampling for this model is **temperature 1.0** and **top_p 0.95**, which is what the cells use and what the checkpoint's `generation_config.json` sets.

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, raise `max_tokens` so the model has budget left to answer after reasoning.

### Client setup

In [3]:
from openai import OpenAI

# Set this to match the served model name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-super"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "null"  # TensorRT-LLM does not require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


def show(response, label=None, show_reasoning=True):
    """Print the answer along with the signals that reveal a bad one."""
    choice = response.choices[0]
    if label:
        print(f"--- {label} ---")

    reasoning = getattr(choice.message, "reasoning", None) or getattr(
        choice.message, "reasoning_content", None
    )
    if show_reasoning and reasoning:
        print("Reasoning:", reasoning)
    if choice.message.content:
        print("Content:", choice.message.content)
    if choice.message.tool_calls:
        print("Tool calls:", choice.message.tool_calls)

    usage = response.usage
    print(
        f"[finish_reason={choice.finish_reason}  "
        f"tokens={usage.prompt_tokens} in / {usage.completion_tokens} out]"
    )
    if choice.finish_reason == "length":
        print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")
    if not (choice.message.content or "").strip() and not choice.message.tool_calls:
        print("WARNING: empty answer - reasoning used the whole budget, raise max_tokens.")


### Single completion

In [4]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
show(response)

Reasoning: The user wants 3 bullet points about TensorRT-LLM. I should provide concise, accurate information about what it is, its key features/benefits, and perhaps its use cases or target hardware.

Key facts about TensorRT-LLM:
- It's an NVIDIA library/framework for optimizing and running inference on large language models (LLMs) on NVIDIA GPUs.
- It provides Python API for defining, optimizing, and executing LLMs.
- Features include quantization (FP8, INT4/INT8), in-flight batching, paged attention, specular decoding (speculative decoding), and support for various model architectures (Llama, Mistral, GPT, etc.).
- It builds on top of TensorRT and FasterTransformer (legacy), enabling high throughput and low latency.
- Supports deployment on NVIDIA GPUs (data center, edge, etc.).

I'll draft 3 bullet points:
1. What it is: NVIDIA's open-source Python library/framework for optimizing and performing high-performance inference of LLMs on NVIDIA GPUs.
2. Key capabilities: Includes advanc

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [5]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=4096,
    )
    show(response, label=f"Q: {prompt}", show_reasoning=False)

--- Q: What is the square root of 144? ---
Content: The square root of 144 is **12**.
[finish_reason=stop  tokens=33 in / 50 out]
--- Q: What is the capital of France? ---
Content: The capital of France is Paris.
[finish_reason=stop  tokens=29 in / 42 out]
--- Q: Explain quantum computing in simple terms. ---
Content: Imagine a regular computer as a huge collection of light switches. Each switch is either **off (0)** or **on (1)**. Every text, photo, and video you handle is just trillions of these switches flipping in patterns.

A quantum computer uses **qubits**. Think of a qubit less like a light switch and more like a **spinning coin**. While it’s spinning, it isn’t clearly heads or tails—it’s in a mix of both at the same time. This is called **superposition**.

When you measure a qubit, it stops “spinning” and becomes a regular 0 or 1. But before that, it can hold many possibilities at once.

Because of this, a few qubits can represent a vast number of combinations all at the same 

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [6]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
    stream=True,
)

section = None
finish_reason = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning_content", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

    if chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason

print(f"\n\n[finish_reason={finish_reason}]")
if finish_reason == "length":
    print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")

Reasoning: The user is asking for the first 5 prime numbers.

    *   A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself.

    *   1: Not prime (by definition).
    *   2: Prime (divisible only by 1 and 2).
    *   3: Prime (divisible only by 1 and 3).
    *   4: Not prime (divisible by 2).
    *   5: Prime (divisible only by 1 and 5).
    *   6: Not prime (divisible by 2 and 3).
    *   7: Prime (divisible only by 1 and 7).
    *   8: Not prime.
    *   9: Not prime.
    *   10: Not prime.
    *   11: Prime.

    *   2, 3, 5, 7, 11.

Content: The first 5 prime numbers are: **2, 3, 5, 7, and 11**.

[finish_reason=stop]


### Reasoning

The model supports two modes: **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`.

In [7]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
show(resp)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about TensorRT-LLM."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
show(resp2)

Reasoning on
Reasoning: *   Topic: GPUs (Graphics Processing Units).
    *   Format: Haiku (5 syllables - 7 syllables - 5 syllables).

    *   Silicon, cores, parallel, threads, rendering, gaming, AI, math, speed, power, heat, chips, memory, CUDA, graphics, pixels.

    Silicon chips shine fast (5)
    Thousands of cores working hard (7)
    Pixels on the screen (5)

    *   Line 1: Si-li-con chips shine fast (5) - OK.
    *   Line 2: Thou-sands of cores work-ing hard (7) - OK.
    *   Line 3: Pix-els on the screen (5) - OK.

    *   Si-li-con cores run fast (5)
    *   Paral-lel math for the game (7)
    *   Pixels light the screen (5)

    *Alternative (more technical/AI focus):*
    Parallel cores hum (5)
    Crunching math for AI dreams (7)
    Silicon power (5)

    *Selection:* The "Parallel cores hum" one is concise and evocative.

    Parallel cores hum, (5)
    Crunching math for AI dreams, (7)
    Silicon power. (5)

    *Check syllables again:*
    Par-al-lel cores hum (5)
 

### Document understanding

The model reads text and structure out of document pages, diagrams, and charts.

The cell below sends a diagram from the [Nemotron 3.5 Lightning NVFP4 blog post](https://developer.nvidia.com/blog/developing-nemotron-3-5-lightning-nvfp4-with-qad-using-nvidia-model-optimizer/) and asks for the labeled values back as JSON, each paired with the exact on-image text it was read from.

> **Note:** Each field is checked against the diagram and printed as PASS or FAIL next to the on-image text it was read from. A FAIL prints the expected value, and a value that is right but poorly cited is called out too. Nothing raises, so the cell always runs to the end.


In [8]:
import json
import re

DOC_IMAGE_URL = (
    "https://developer-blogs.nvidia.com/wp-content/uploads/2026/08/"
    "nvfp4-qad-nemotron-3.5-lightning.png"
)

EXPECTED = {
    "title": "QAD for Nemotron 3.5 Lightning NVFP4",
    "step_1_name": "Post-Training Quantization",
    "step_2_name": "Distillation",
    "teacher_precision": "BF16",
    "teacher_size": "66 GB",
    "student_size": "22 GB",
}
FIELDS = list(EXPECTED)


def norm(text):
    """Compare on characters only, so '66 GB' and '66GB' both match."""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


prompt = (
    "Read the diagram and return only JSON, no prose, in this exact shape:\n"
    '{"fields": {' + ", ".join(f'"{f}": "..."' for f in FIELDS) + "}, "
    '"citations": {' + ", ".join(f'"{f}": "..."' for f in FIELDS) + "}}\n"
    "Each citation must be the exact text you read that value from in the image."
)

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": DOC_IMAGE_URL}},
            ],
        }
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

choice = resp.choices[0]
print(
    f"finish_reason: {choice.finish_reason}  |  "
    f"tokens: {resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out"
)

raw = re.sub(r"^```(?:json)?|```$", "", (choice.message.content or "").strip(), flags=re.M)

try:
    result = json.loads(raw.strip())
except json.JSONDecodeError:
    print(raw)
else:
    fields = result.get("fields", {})
    citations = result.get("citations", {})
    for f in FIELDS:
        value = fields.get(f, "")
        citation = citations.get(f) or ""
        want = norm(EXPECTED[f])
        ok = want in norm(value)
        print(f"  {'PASS' if ok else 'FAIL'}  {f:<18} {str(value) or '-'}")
        print(f"        read from {citation!r}")
        if not ok:
            print(f"        expected {EXPECTED[f]!r}")
        elif not citation:
            print("        no citation returned for this value")
        elif want not in norm(citation):
            print("        value matches but its citation does not contain it")


finish_reason: stop  |  tokens: 1520 in / 168 out
  title                  QAD for Nemotron 3.5 Lightning NVFP4 <- 'QAD for Nemotron 3.5 Lightning NVFP4'
  step_1_name            Post-Training Quantization <- 'STEP 1 Post-Training Quantization'
  step_2_name            Distillation <- 'STEP 2 Distillation'
  teacher_precision      BF16         <- 'Teacher BF16'
  teacher_size           66 GB        <- 'Teacher BF16 66 GB'
  student_size           22 GB        <- 'Student NVFP4 22 GB'


### Video understanding

The cell below sends a short NVIDIA Omniverse clip to the model via the OpenAI-compatible API and asks it to describe the scene. Use MP4 clips of 1-2 minutes.

Frame sampling is a server-launch setting.

> **Note:** Try video prompts both with and without thinking enabled, since which one works better varies by clip. The cell below disables it for a faster, more direct answer.

In [9]:
video_url = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what happens in this video."},
                {"type": "video_url", "video_url": {"url": video_url}},
            ],
        }
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

show(resp)

Content: The video displays a series of frames showing a digital environment with a snowy landscape, trees, and a small house. The house appears to be made of wood with a sloped roof, and it is surrounded by snow-covered ground and trees. The color palette is predominantly white due to the snow, with shades of brown from the house and green from the trees. There are no visible characters or moving objects, and the environment is static throughout the frames.
[finish_reason=stop  tokens=1426 in / 92 out]


### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`, then run the function and hand its result back so the model can answer the user.

The chat template emits tool calls as `<tool_call><function=name><parameter=...>` XML, which is what the `qwen3_coder` tool parser converts into OpenAI `tool_calls`.

> **Note:** When tool calling with reasoning enabled, pass `"force_nonempty_content": true` inside `chat_template_kwargs`.

In [10]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calculate the tip amount for a bill",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                "tip_percentage": {"type": "integer", "description": "The percentage of tip to apply"},
            },
            "required": ["bill_total", "tip_percentage"],
        },
    },
}]

messages = [{"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "force_nonempty_content": True}},
)

choice = completion.choices[0]
show(completion)

Reasoning: The user wants to calculate a tip for a bill of $50 at 15%.
I should use the `calculate_tip` function.
Parameters:
- `bill_total`: 50
- `tip_percentage`: 15

Let's call the function.

Content: 


Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-4f9ffc4614e849d9a9136c5d6251078f', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]
[finish_reason=tool_calls  tokens=346 in / 100 out]


In [11]:
import json

def calculate_tip(bill_total, tip_percentage):
    """Return the tip and the final total for a bill."""
    tip = round(bill_total * tip_percentage / 100, 2)
    return {"tip": tip, "total": round(bill_total + tip, 2)}

# Map schema names to real functions, then run whichever one the model picked
tool_functions = {"calculate_tip": calculate_tip}

call = choice.message.tool_calls[0]
result = tool_functions[call.function.name](**json.loads(call.function.arguments))

# Hand the result back so the model can answer the user
followup = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages + [
        choice.message,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ],
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Tool result:", result)
show(followup)

Tool result: {'tip': 7.5, 'total': 57.5}
Content: For a $50 bill, a 15% tip would be **$7.50**, making your total **$57.50**.
[finish_reason=stop  tokens=479 in / 34 out]


### Controlling reasoning budget

`ThinkingBudgetClient` below caps how long the model reasons. It asks for at most `reasoning_budget` tokens of reasoning, closes the reasoning trace if the model has not closed it, then sends a second request for the answer using the tokens left over. If the model already answered within the reasoning budget, that answer is returned as is and no second request is sent.

> **Note:** `max_tokens` must be larger than `reasoning_budget`, since the answer is generated from the remainder.

In [13]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name_or_path, trust_remote_code=True
        )
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        choice = response.choices[0]
        reasoning_content = choice.message.reasoning_content or ""

        # The model may finish inside the reasoning budget - return that answer
        # rather than spending a second request regenerating it
        if choice.finish_reason == "stop" and (choice.message.content or "").strip():
            return {
                "reasoning_content": reasoning_content.strip(),
                "content": choice.message.content,
                "finish_reason": choice.finish_reason,
            }

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # Copy before extending so the caller's list is left untouched
        messages = list(messages) + [{"role": "assistant", "content": reasoning_content}]
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().removesuffix("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [14]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4"  # use actual HF model ID for tokenizer
)

In [15]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])
print(f"\n[finish_reason={resp['finish_reason']}]")
if resp["finish_reason"] == "length":
    print("WARNING: cut off by max_tokens - raise max_tokens and re-run.")

Reasoning: The user wants a haiku about GPUs. A haiku is a 3-line poem with a 5-7-5 syllable structure.

 Topic: GPUs (Graphics Processing Units)

 Let's draft:

 Line 1 (5 syllables): Silicon cores shimmer (5)
 Line 2 (7 syllables): Parallel threads paint the screen (7)
 Line 3 (5 syllables): Fast math, bright colors (5)

 Let's check syllables:
 Si-li-con cores shim-mer = 5
 Par-al-lel threads paint the screen = 7
 Fast math, bright col-ors = 5

 Or maybe.
Content: Silicon cores shine,
Parallel threads paint the screen,
Fast math, bright colors.

[finish_reason=stop]


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).